# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(url)

metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Authors: {[a['@id'] for a in metadata.author] if hasattr(metadata, 'author') else 'No author listed'}")
print(f"Dataset ID: {metadata.identifier}")
print(f"RecordSets: {metadata.recordSet}")
print(f"Fields (personal sensitive info): {metadata.personalSensitiveInformation}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

**Tip:** In Croissant, each entity is referenced by its unique `@id`.

In [ ]:
# List the record sets and their fields using @id
record_sets = dataset.record_sets
print("Available RecordSets:")
record_sets_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} - name: {rs.get('name', 'Unknown')}")
    record_sets_ids.append(rs['@id'])
    print("Fields:")
    for field in rs.get('field', []):
        print(f"    Field @id: {field['@id']} - name: {field.get('name', 'Unknown')}")
    print("")
# Show all available @ids
print(f"All record set @ids: {record_sets_ids}")

# Explore records from a record set (using @id; taking the first record set as example)
example_record_set_id = record_sets_ids[0] if record_sets_ids else None
if example_record_set_id:
    print(f"Sample records from: {example_record_set_id}")
    for x in dataset.records(record_set=example_record_set_id):
        print(x)
        break

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

All entities are referenced via their `@id`.

In [ ]:
# Extract data for each record set
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
    print(df.head(2))

# Pick a RecordSet to analyze further
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Fields for RecordSet {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

**Refer to fields and columns by their `@id`.**

In [ ]:
# Identify available numeric fields by @id
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# Select 'PHQ-9 score' column for demonstration (assume its @id is 'phq9_score')
# Replace 'phq9_score' with actual @id from previous cell if different
numeric_field_id = numeric_fields[0] if numeric_fields else None

# Set threshold for filtering
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., 'level_of_education')
    group_field_id = None
    for col in df.columns:
        if 'education' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

Here, we plot the distribution of a numeric field (e.g., PHQ-9 score) and show group means by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health survey dataset from Kilifi County, Kenya using `mlcroissant`.

We:
- Loaded Croissant schema metadata and examined record sets and fields by `@id`.
- Extracted data into Pandas DataFrames.
- Filtered and normalized numeric fields referenced by their `@id`.
- Grouped data by key attributes (education level).
- Visualized distributions and grouped means.

For further analysis, reference fields by their `@id` to ensure consistency and reproducibility.